# `json2vec` Hello World

This notebook trains a tiny model from an in-memory synthetic dataset.

In [1]:
import lightning.pytorch as lit
import torch
from rich.pretty import pprint

import json2vec as j2v

In [ ]:
@j2v.shim(yields=True)
def hello_world_records(observation: dict, strata: j2v.Strata):

    records = [
        {"color": "red", "label": "warm"},
        {"color": "orange", "label": "warm"},
        {"color": "yellow", "label": "warm"},
        {"color": "blue", "label": "cool"},
        {"color": "green", "label": "cool"},
        {"color": "purple", "label": "cool"},
    ]

    yield from records

In [ ]:
params = j2v.Hyperparameters(
    d_model=16,
    fields=j2v.Array(
        name="record",
        fields=[
            j2v.Category(name="color", query="[*].color", max_vocab_size=16),
            j2v.Category(name="label", query="[*].label", max_vocab_size=8, topk=[2]),
        ],
    ),
    target=j2v.Address("record", "label"),
    embed=j2v.Address("record"),
)


model = j2v.Architecture(
    hyperparameters=params,
    batch_size=4,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.StreamingDataModule.from_model(
    model,
    dataset=j2v.Dataset(processor=hello_world_records),
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    file_buffer_size=1,
    observation_buffer_size=32,
    sample_rate=1.0,
)

2026-05-17 19:51:32.309 | INFO     | json2vec.architecture.root:__init__:128 - initialized JSON2Vec module


In [ ]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=20,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/grantham/Desktop/json2vec-oss/examples/checkpoints exists and is not empty.
2026-05-17 19:51:32.340 | INFO     | json2vec.data.datasets:da

In [5]:
batch = [[{"color": "red"}], [{"color": "blue"}]]

In [6]:
pprint(model.predict(batch))

{
│   'record/label': {
│   │   'state': {
│   │   │   'valued': [0.9962130188941956, 0.9869213104248047],
│   │   │   'null': [0.0014335822779685259, 0.004264552611857653],
│   │   │   'padded': [0.0006813588552176952, 0.0032666651532053947],
│   │   │   'masked': [0.0011903889244422317, 0.0036857472732663155],
│   │   │   'other': [0.0004817021545022726, 0.0018617414170876145]
│   │   },
│   │   'content': {
│   │   │   'value': ['warm', 'cool'],
│   │   │   'probability': [0.9881111979484558, 0.9842490553855896],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'warm', 'probability': 0.9881111979484558},
│   │   │   │   │   {'label': 'cool', 'probability': 0.011888709850609303}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'cool', 'probability': 0.9842490553855896},
│   │   │   │   │   {'label': 'warm', 'probability': 0.015750806778669357}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

In [7]:
pprint(model.embed(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   -0.19812482595443726,
│   │   │   │   0.01926981657743454,
│   │   │   │   0.16341471672058105,
│   │   │   │   -0.2151465117931366,
│   │   │   │   -0.058021243661642075,
│   │   │   │   0.1820082664489746,
│   │   │   │   0.46504586935043335,
│   │   │   │   -0.34611931443214417,
│   │   │   │   -0.27737176418304443,
│   │   │   │   -0.04056933522224426,
│   │   │   │   0.30515310168266296,
│   │   │   │   -0.01910339668393135,
│   │   │   │   -0.19576993584632874,
│   │   │   │   0.18579593300819397,
│   │   │   │   -0.40196743607521057,
│   │   │   │   0.3291427195072174
│   │   │   ],
│   │   │   [
│   │   │   │   0.06469237059354782,
│   │   │   │   -0.2544524669647217,
│   │   │   │   -0.13366061449050903,
│   │   │   │   0.2574854791164398,
│   │   │   │   -0.2721704840660095,
│   │   │   │   -0.06137020140886307,
│   │   │   │   -0.13495077192783356,
│   │   │   │   0.051571477204561234,
│   │   │   │   0.33312278985977173,
│   │   │   │   0.2194756418466568,
│   │   │   │   -0.21267323195934296,
│   │   │   │   0.1814790517091751,
│   │   │   │   -0.13751277327537537,
│   │   │   │   0.25877413153648376,
│   │   │   │   0.37207844853401184,
│   │   │   │   -0.5353217124938965
│   │   │   ]
│   │   ]
│   }
}